# [5.6] Multimodal, Embedding, and Function-Calling Models - Solutions

Reference validation notebook for the section-local embedding, probe, tool-schema, parser, and attribution implementation. This executes the visible tests against `solutions.py`, then checks the CPU notebook contract and the committed CUDA report highlights.

Expected CUDA highlights: pinned public BGE retrieval passes with a permuted-pair negative control, direct authenticated EmbeddingGemma retrieval passes with matching controls, pinned public FunctionGemma Mobile Actions generation passes parse/name thresholds on the deterministic held-out slice, direct authenticated base FunctionGemma loading passes a benign forward check, and peak VRAM stays below the configured budget.


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter5_modern_architectures"
section = "part6_multimodal_embedding_function_models"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_multimodal_embedding_function_models.tests as tests
from part6_multimodal_embedding_function_models import solutions


In [ ]:
tests.test_mean_pool_embeddings_ignores_padding_and_matches_reference(
    solutions.mean_pool_embeddings,
)
tests.test_retrieval_metrics_rank_pairs_and_hard_negative_margin(
    solutions.cosine_similarity_matrix,
    solutions.retrieval_ranks,
    solutions.embedding_retrieval_report,
)
tests.test_centroid_probe_recovers_heldout_clusters(
    solutions.fit_centroid_probe,
    solutions.predict_centroid_probe,
    solutions.centroid_probe_accuracy,
)
tests.test_mask_disallowed_tools_blocks_invalid_logits(solutions.mask_disallowed_tools)
tests.test_function_call_report_separates_tool_and_abstention_errors(
    solutions.function_call_report,
)
tests.test_parse_function_call_text_extracts_name_and_arguments(
    solutions.parse_function_call_text,
)
tests.test_schema_token_attribution_matches_dot_products(
    solutions.schema_token_attribution,
)
tests.test_notebook_contract(solutions.run_smoke_test)


In [ ]:
smoke = solutions.run_smoke_test(cpu=True)
smoke


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["bge_preflight_passed"], "BGE retrieval preflight should pass."
assert gpu["bge_permuted_control_fails"], "Permuted retrieval control should fail top-1."
assert gpu["functiongemma_preflight_passed"], "FunctionGemma Mobile Actions preflight should pass."
assert gpu["functiongemma_parse_accuracy"] == 1.0, "FunctionGemma generations should parse on the eval slice."
assert gpu["functiongemma_function_name_accuracy"] == 1.0, "Function names should match the held-out labels."
assert gpu["functiongemma_required_argument_accuracy"] >= 0.85, "Required argument accuracy should clear the release threshold."
assert gpu["embeddinggemma_preflight_passed"], "Direct EmbeddingGemma retrieval preflight should pass."
assert gpu["embeddinggemma_ready_for_direct_loading"], "EmbeddingGemma should be locally authenticated and ready."
assert gpu["functiongemma_base_preflight_passed"], "Base FunctionGemma CUDA forward preflight should pass."
assert gpu["functiongemma_base_ready_for_direct_loading"], "Base FunctionGemma should be locally authenticated and ready."
{
    "bge_top1": gpu["bge_retrieval_top1_accuracy"],
    "bge_margin": gpu["bge_mean_margin"],
    "functiongemma_required_argument_accuracy": gpu["functiongemma_required_argument_accuracy"],
    "functiongemma_failure_count": gpu["functiongemma_failure_count"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
